# Sri Lanka Heat IBF — Bayesian Warning System Demo
## Colombo District Pilot · CREWS SA · 2026

This notebook demonstrates the full pipeline of the **Bayesian Impact-Based Heat Warning System** developed for the Sri Lanka National Demonstration Plan. It covers:

1. **Data loading** — synthetic seasonal cycle, built-in demo, or your own CSV
2. **Model calibration** — climatological, calibrated Bayesian, raw ensemble
3. **Warning decisions** — optimal thresholds from decision theory
4. **Sector actions** — Health, Agriculture, Education, Labour (Rogers 2026)
5. **Verification** — Brier scores, POD, FAR, CSI, cumulative loss curves
6. **Single-forecast advisory** — ready-to-issue output

**Theoretical foundation:** Economou T, Stephenson DB et al. (2016) *On the use of Bayesian decision theory for issuing natural hazard warnings*. Proc. R. Soc. A 472: 20160295

---
## 0 · Setup

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib
from matplotlib.gridspec import GridSpec

# Adjust path if notebook is in a different directory from heat_ews_srilanka.py
sys.path.insert(0, os.path.dirname(os.path.abspath('heat_ews_srilanka.py')))

from heat_ews_srilanka import (
    HeatData, HeatWarningSystem,
    ClimatologicalModel, CalibratedModel, EnsembleModel,
    build_loss_matrix, bayes_warning, brier_scores, hit_rates,
    generate_demo_data, categorise_hi,
    STATE_LABELS, WARNING_LABELS, HI_THRESHOLDS,
    SECTOR_LOSS_PARAMS, SECTOR_ACTIONS
)

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 110,
    'figure.facecolor': 'white',
    'axes.facecolor': '#FAFAFA',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
})

# Colour scheme
COLORS = {
    1: ('#2E7D32', '#E8F5E9'),
    2: ('#F9A825', '#FFFDE7'),
    3: ('#E65100', '#FFF3E0'),
    4: ('#B71C1C', '#FFEBEE'),
}
WARN_SHORT = {1: 'No Warning', 2: 'Watch', 3: 'Alert', 4: 'Warning'}
SECTOR_COLORS = {'health': '#C62828', 'agri': '#2E7D32', 'labour': '#E65100', 'balanced': '#1565C0'}

print('Setup complete.')

---
## 1 · Data

Three data modes are available. **Uncomment the one you want.**

| Mode | When to use |
|---|---|
| Built-in synthetic | Quick start — no files needed |
| Seasonal synthetic | Shows realistic Feb–Apr heat peak |
| Your own CSV | Real Colombo observations from DoM |

### CSV format
If you use your own data, the CSV must have these columns:

| Column | Required | Description |
|---|---|---|
| `date` | ✅ | ISO format YYYY-MM-DD |
| `heat_index` | ✅ | Observed Heat Index in °C |
| `forecast_p1` … `forecast_p4` | Optional | Ensemble forecast probabilities for each severity category |

In [ ]:
# ── MODE 1: Built-in synthetic ──────────────────────────────
train_data, test_data = generate_demo_data(n_train=180, n_test=90)
DATA_MODE = 'Built-in synthetic'
hi_signal = None
dates = None

# ── MODE 2: Seasonal synthetic (uncomment to use) ───────────
# import datetime
# def generate_seasonal_data(seed=42):
#     rng = np.random.default_rng(seed)
#     n = 365
#     doy = np.arange(1, n+1)
#     hi = (38.0
#           + 8.0*np.sin(2*np.pi*(doy-60)/365)
#           + 4.0*np.exp(-0.5*((doy-100)/20)**2)
#           - 3.0*np.exp(-0.5*((doy-200)/40)**2)
#           + rng.normal(0, 2, n))
#     obs = np.array([categorise_hi(float(v)) for v in hi])
#     probs = np.zeros((n, 4))
#     for t in range(n):
#         x = obs[t]; base = np.zeros(4); base[x-1]=0.55; base+=0.12
#         probs[t] = rng.dirichlet(base*6+1)
#     base_date = datetime.date(2026, 1, 1)
#     dates_ = [(base_date + datetime.timedelta(days=i)).isoformat() for i in range(n)]
#     return HeatData(obs[:270], probs[:270], dates_[:270]), HeatData(obs[270:], probs[270:], dates_[270:]), hi, dates_
# train_data, test_data, hi_signal, dates = generate_seasonal_data()
# DATA_MODE = 'Seasonal synthetic'

# ── MODE 3: Your own CSV (uncomment and set path) ───────────
# import pandas as pd
# CSV_PATH = 'colombo_observations.csv'
# df = pd.read_csv(CSV_PATH)
# obs_cats = np.array([categorise_hi(float(h)) for h in df['heat_index']])
# dates = df['date'].astype(str).tolist()
# has_fc = all(f'forecast_p{j}' in df.columns for j in range(1,5))
# if has_fc:
#     probs = df[['forecast_p1','forecast_p2','forecast_p3','forecast_p4']].values.astype(float)
#     probs = probs / probs.sum(axis=1, keepdims=True)
# else:
#     rng = np.random.default_rng(0); probs = np.zeros((len(obs_cats),4))
#     for t in range(len(obs_cats)):
#         x=obs_cats[t]; b=np.zeros(4); b[x-1]=0.5; b+=0.12
#         probs[t]=rng.dirichlet(b*5+1)
# split = int(len(obs_cats)*0.75)
# train_data = HeatData(obs_cats[:split], probs[:split], dates[:split])
# test_data  = HeatData(obs_cats[split:], probs[split:], dates[split:])
# DATA_MODE = f'CSV: {CSV_PATH}'

print(f'Data mode  : {DATA_MODE}')
print(f'Training   : {train_data.n} days')
print(f'Evaluation : {test_data.n} days')
print(f'\nObserved category distribution (training):')
for j in range(1,5):
    n = (train_data.observed_categories == j).sum()
    pct = 100*n/train_data.n
    bar = '█'*int(pct/2)
    print(f'  Cat {j} {STATE_LABELS[j]}: {n:3d} days ({pct:5.1f}%) {bar}')

---
## 2 · Heat Index Thresholds

Sri Lanka DoM currently uses these Heat Index thresholds. They will be refined using the Bayesian calibration in Section 4.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 2.5))
ax.set_xlim(25, 65)
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('Sri Lanka Heat Index Threshold Scale (DoM 2018 — under review for IBF pilot)',
             fontsize=12, fontweight='bold', pad=12)

ranges = [(25,39,'Normal','#2E7D32'),
          (39,46,'Caution','#F9A825'),
          (46,52,'Extreme\nCaution','#E65100'),
          (52,65,'Danger','#B71C1C')]

for lo, hi, label, color in ranges:
    width = hi - lo
    ax.barh(0.5, width, left=lo, height=0.55, color=color, alpha=0.85)
    ax.text(lo + width/2, 0.5, label, ha='center', va='center',
            color='white', fontweight='bold', fontsize=10)
    if lo > 25:
        ax.text(lo, 0.07, f'{lo}°C', ha='center', va='bottom', fontsize=9, color='#333')

ax.text(25, 0.07, '25°C', ha='center', va='bottom', fontsize=9, color='#333')
ax.text(63, 0.07, '65°C+', ha='center', va='bottom', fontsize=9, color='#333')

# Warning level labels
for (lo,hi,_,_), label, warn_color in zip(ranges,
        ['No Warning','Heat Watch','Heat Alert','Heat Warning'],
        ['#2E7D32','#7B5800','#E65100','#B71C1C']):
    ax.text(lo + (hi-lo)/2, 0.95, label, ha='center', va='top',
            fontsize=8, color=warn_color, style='italic')

plt.tight_layout()
plt.show()

---
## 3 · Loss Function — Sector Profiles

The Bayesian decision rule minimises **expected loss**. Different sectors have different costs of acting (false alarms) vs. not acting (missed events). The loss function parameters below are drawn from **Rogers (2026)** prescriptive models.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4), sharey=True)
fig.suptitle('Loss Matrix L(warning level, observed state) — by sector\n'
             'Higher = worse outcome. Optimal warning should minimise expected value.',
             fontsize=12, fontweight='bold')

cat_labels = ['Normal','Caution','Extreme','Danger']
warn_labels = ['Green','Yellow','Amber','Red']

for ax, (sector, params) in zip(axes, SECTOR_LOSS_PARAMS.items()):
    L = build_loss_matrix(*params)
    im = ax.imshow(L, cmap='RdYlGn_r', vmin=0, vmax=110, aspect='auto')
    ax.set_xticks(range(4)); ax.set_xticklabels(cat_labels, rotation=30, ha='right', fontsize=9)
    ax.set_yticks(range(4)); ax.set_yticklabels(warn_labels, fontsize=9)
    ax.set_title(sector.capitalize(), fontsize=11, fontweight='bold',
                 color=SECTOR_COLORS[sector])
    for i in range(4):
        for j in range(4):
            ax.text(j, i, str(L[i,j]), ha='center', va='center',
                    fontsize=10, fontweight='bold',
                    color='white' if L[i,j] > 60 else '#222')

plt.colorbar(im, ax=axes[-1], label='Loss (relative units)', shrink=0.8)
axes[0].set_ylabel('Issued warning level')
plt.tight_layout()
plt.show()

---
## 4 · Model Calibration

We fit three models to the training data:
- **Climatological (CLIM):** ignores the forecast, uses base rates only
- **Raw Ensemble (ENS):** uses forecast probabilities directly
- **Calibrated Bayesian (CAL):** applies Bayes' theorem to correct forecast biases

In [ ]:
clim = ClimatologicalModel().fit(train_data)
cal  = CalibratedModel().fit(train_data)
ens  = EnsembleModel()

print('Climatological p(x):  ', np.round(clim.p_x, 3))
print('\nCalibrated contingency table (rows=forecast modal cat, cols=observed cat):')
print(np.round(cal.contingency, 0).astype(int))
print('\nModels fitted. Proceed to Section 5 for verification.')

In [ ]:
# Visualise how each model responds to a changing forecast
p_danger_scan = np.linspace(0, 0.6, 40)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Model Calibration — P(Extreme+Danger) output vs forecast input',
             fontsize=13, fontweight='bold')

ax = axes[0]
for model, label, color, ls in [
    (clim, 'Climatological', '#90A4AE', ':'),
    (cal,  'Calibrated (CAL)', '#1565C0', '-'),
    (ens,  'Raw Ensemble (ENS)', '#E65100', '--'),
]:
    ph = []
    for p4 in p_danger_scan:
        p3 = min(p4 * 1.5, 0.4)
        p1 = max(0, (1 - p3 - p4) * 0.6)
        p2 = max(0, 1 - p1 - p3 - p4)
        fc = np.array([p1, p2, p3, p4]); fc /= fc.sum()
        if hasattr(model, 'predict_from_z'):
            ph.append(model.predict(fc)[2:].sum())
        else:
            ph.append(model.predict(fc)[2:].sum())
    ax.plot(p_danger_scan, ph, label=label, color=color, linestyle=ls, linewidth=2)

ax.plot(p_danger_scan, p_danger_scan * 2.5, color='grey', linestyle=':',
        linewidth=1, label='Reference (raw input)')
ax.set_xlabel('Input P(Danger) in forecast', fontsize=11)
ax.set_ylabel('Output P(Extreme + Danger)', fontsize=11)
ax.set_title('How much does each model\namplify or dampen the danger signal?', loc='left')
ax.legend()
ax.set_ylim(0, 1)

# Panel 2: warning level switching
ax2 = axes[1]
L_bal = build_loss_matrix(*SECTOR_LOSS_PARAMS['balanced'])
for model, label, color, ls in [
    (clim, 'Climatological', '#90A4AE', ':'),
    (cal,  'Calibrated (CAL)', '#1565C0', '-'),
    (ens,  'Raw Ensemble (ENS)', '#E65100', '--'),
]:
    warn_levels = []
    for p4 in p_danger_scan:
        p3 = min(p4*1.5, 0.4)
        p1 = max(0, (1-p3-p4)*0.6); p2 = max(0, 1-p1-p3-p4)
        fc = np.array([p1,p2,p3,p4]); fc/=fc.sum()
        px = model.predict(fc)
        warn_levels.append(bayes_warning(px, L_bal))
    ax2.step(p_danger_scan, warn_levels, where='mid',
             label=label, color=color, linestyle=ls, linewidth=2.5)

ax2.set_yticks([1,2,3,4])
ax2.set_yticklabels([WARN_SHORT[w] for w in [1,2,3,4]])
ax2.set_xlabel('Input P(Danger) in forecast', fontsize=11)
ax2.set_ylabel('Optimal warning level')
ax2.set_title('Warning switching point\n(balanced sector loss)', loc='left')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)
ax2.set_ylim(0.5, 4.5)

plt.tight_layout()
plt.show()

---
## 5 · Verification — Brier Scores, POD, FAR, CSI

The Brier score measures probabilistic accuracy (lower = better). POD, FAR and CSI measure warning decision quality for high-heat events (Cat 3+4).

In [ ]:
results = {}
for model_type in ['climatological', 'calibrated', 'ensemble']:
    sys_ = HeatWarningSystem(sector='balanced', model_type=model_type)
    if model_type != 'ensemble': sys_.fit(train_data)
    r = sys_.evaluate(test_data)
    bs = brier_scores(test_data, sys_.model)
    hr = hit_rates(test_data, sys_.loss_matrix, sys_.model)
    results[model_type] = {'r': r, 'bs': bs, 'hr': hr}

print(f'{"Model":25s} {"TotLoss":>10} {"BrierMean":>10} {"POD":>7} {"FAR":>7} {"CSI":>7}')
print('-'*70)
for mt, v in results.items():
    print(f'{mt:25s} {v["r"]["total_loss"]:10.0f} '
          f'{v["bs"].mean():10.4f} {v["hr"]["POD"]:7.2f} '
          f'{v["hr"]["FAR"]:7.2f} {v["hr"]["CSI"]:7.2f}')

In [ ]:
cats = ['Normal\n(Cat 1)', 'Caution\n(Cat 2)', 'Extreme\n(Cat 3)', 'Danger\n(Cat 4)']
models_list = list(results.keys())
model_labels_list = ['Climatological', 'Calibrated (CAL)', 'Raw Ensemble']
model_colors_list = ['#90A4AE', '#1565C0', '#E65100']
x = np.arange(4); width = 0.25

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Verification — Evaluation Period', fontsize=13, fontweight='bold')

# Brier scores
ax = axes[0]
for i, (mt, label, color) in enumerate(zip(models_list, model_labels_list, model_colors_list)):
    ax.bar(x + i*width, results[mt]['bs'], width, label=label, color=color, alpha=0.85)
ax.set_xticks(x + width); ax.set_xticklabels(cats, fontsize=9)
ax.set_ylabel('Brier Score (lower = better)')
ax.set_title('Brier Score by category', loc='left')
ax.legend(fontsize=9)
ax.axhline(0.25, color='grey', linewidth=0.8, linestyle=':', label='Random')

# POD
ax2 = axes[1]
pod_vals = [results[mt]['hr']['POD'] for mt in models_list]
bars = ax2.bar(model_labels_list, pod_vals, color=model_colors_list, alpha=0.85)
ax2.set_ylabel('POD')
ax2.set_title('Probability of Detection\n(high-heat events, Cat 3+4)', loc='left')
ax2.set_ylim(0, 1.15)
for bar, val in zip(bars, pod_vals):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
             f'{val:.2f}', ha='center', fontweight='bold')

# FAR
ax3 = axes[2]
far_vals = [results[mt]['hr']['FAR'] for mt in models_list]
bars2 = ax3.bar(model_labels_list, far_vals, color=model_colors_list, alpha=0.85)
ax3.set_ylabel('FAR')
ax3.set_title('False Alarm Rate\n(lower = fewer false alarms)', loc='left')
ax3.set_ylim(0, 1.15)
ax3.axhline(0.30, color='#F9A825', linestyle='--', linewidth=1.5, label='Target FAR < 0.30')
ax3.legend(fontsize=9)
for bar, val in zip(bars2, far_vals):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
             f'{val:.2f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 6 · Cumulative Loss Over the Evaluation Period

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Cumulative Loss — Evaluation Period (lower = better)', fontsize=13, fontweight='bold')

# Model comparison
ax = axes[0]
for model_type, label, color, ls in [
    ('climatological', 'Climatological', '#90A4AE', ':'),
    ('ensemble',       'Raw Ensemble',   '#F9A825', '--'),
    ('calibrated',     'Calibrated (CAL)', '#1565C0', '-'),
]:
    sys_ = HeatWarningSystem(sector='balanced', model_type=model_type)
    if model_type != 'ensemble': sys_.fit(train_data)
    r = sys_.evaluate(test_data)
    ax.plot(np.cumsum(r['actual_losses']), label=label, color=color, linestyle=ls, linewidth=2)
ax.set_xlabel('Day in evaluation period')
ax.set_ylabel('Cumulative loss')
ax.set_title('Model comparison\n(balanced sector)', loc='left')
ax.legend()

# Sector comparison
ax2 = axes[1]
for sector, color in SECTOR_COLORS.items():
    sys_ = HeatWarningSystem(sector=sector, model_type='calibrated')
    sys_.fit(train_data)
    r = sys_.evaluate(test_data)
    ax2.plot(np.cumsum(r['actual_losses']), label=sector.capitalize(), color=color, linewidth=2)
ax2.set_xlabel('Day in evaluation period')
ax2.set_ylabel('Cumulative loss')
ax2.set_title('Sector comparison\n(calibrated model)', loc='left')
ax2.legend()

plt.tight_layout()
plt.show()

---
## 7 · Warning Calendar — Evaluation Period

In [ ]:
sys_cal = HeatWarningSystem(sector='balanced', model_type='calibrated')
sys_cal.fit(train_data)
r = sys_cal.evaluate(test_data)
issued = np.array(r['warning_issued'], dtype=int)
n = test_data.n

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True,
                          gridspec_kw={'height_ratios': [1.5, 1, 1]})
fig.suptitle('Warning Calendar — Evaluation Period', fontsize=13, fontweight='bold')

# Warning colours
ax = axes[0]
for t in range(n):
    ax.bar(t, 1, color=COLORS[issued[t]][0], edgecolor='none')
ax.set_yticks([])
ax.set_ylabel('Warning level')
patches = [mpatches.Patch(color=COLORS[w][0], label=WARN_SHORT[w]) for w in [1,2,3,4]]
ax.legend(handles=patches, ncol=4, loc='upper right')
ax.set_title('Issued warning (calibrated Bayesian)', loc='left')

# P(high heat)
ax2 = axes[1]
p_high = test_data.forecast_probs[:,2] + test_data.forecast_probs[:,3]
ax2.fill_between(range(n), 0, p_high, color='#E65100', alpha=0.4)
ax2.axhline(0.25, color='#F9A825', linestyle='--', linewidth=1.2, label='Watch threshold')
ax2.axhline(0.50, color='#B71C1C', linestyle='--', linewidth=1.2, label='Alert threshold')
ax2.set_ylim(0, 1); ax2.set_ylabel('P(Cat 3+4)')
ax2.legend(fontsize=9); ax2.set_title('Forecast: P(Extreme + Danger)', loc='left')

# Observed category
ax3 = axes[2]
for t in range(n):
    c = test_data.observed_categories[t]
    ax3.bar(t, c, color=COLORS[c][0], alpha=0.8, edgecolor='none')
ax3.set_yticks([1,2,3,4])
ax3.set_yticklabels(['Normal','Caution','Extreme','Danger'], fontsize=9)
ax3.set_xlabel('Day in evaluation period'); ax3.set_ylabel('Observed category')
ax3.set_title('Observed heat severity', loc='left')

plt.tight_layout()
plt.show()

---
## 8 · Single-Forecast Advisory

This is the operational output: given a forecast probability vector for tomorrow, what is the optimal warning level and what actions should each sector take?

**Try changing the forecast values below** to explore how the system responds.

In [ ]:
# ── Enter your forecast here ──────────────────────────────────
# Probabilities for [Normal, Caution, Extreme Caution, Danger]
# They will be auto-normalised to sum to 1.

FORECAST = [0.10, 0.30, 0.40, 0.20]   # ← change these

# Or convert from a Heat Index point forecast:
# HI_FORECAST = 48.5  # °C
# cat = categorise_hi(HI_FORECAST)
# FORECAST = [0.05, 0.15, 0.55, 0.25]  # typical ensemble around cat 3

# ── Issue warnings for all sectors ───────────────────────────
print('FORECAST PROBABILITIES')
for j, (label, p) in enumerate(zip(['Normal','Caution','Extreme','Danger'], FORECAST)):
    total = sum(FORECAST)
    pct = 100*p/total
    bar = '█'*int(pct/3)
    print(f'  Cat {j+1} ({label:8s}): {pct:5.1f}%  {bar}')

print()
print(f'{"Sector":12s} | {"Warning Level":25s} | {"Min Expected Loss":>18}')
print('-'*62)
for sector, color in SECTOR_COLORS.items():
    sys_ = HeatWarningSystem(sector=sector, model_type='calibrated')
    sys_.fit(train_data)
    result = sys_.issue_warning(FORECAST)
    print(f'{sector:12s} | {result["warning_label"]:25s} | {result["min_expected_loss"]:18.1f}')

In [ ]:
# Full advisory panel for the balanced-sector decision
sys_bal = HeatWarningSystem(sector='balanced', model_type='calibrated')
sys_bal.fit(train_data)
result = sys_bal.issue_warning(FORECAST)
w = result['warning']

fig = plt.figure(figsize=(12, 7))
fig.patch.set_facecolor(COLORS[w][1])
gs = GridSpec(2, 3, figure=fig, hspace=0.5, wspace=0.4)
fig.suptitle('Sri Lanka Heat IBF — Advisory Output (Balanced Sector)',
             fontsize=14, fontweight='bold')

# Warning box
ax0 = fig.add_subplot(gs[0, :2])
ax0.set_facecolor(COLORS[w][0])
ax0.text(0.5, 0.55, WARNING_LABELS[w].upper(), transform=ax0.transAxes,
         fontsize=20, fontweight='bold', color='white', ha='center', va='center')
ax0.text(0.5, 0.18, 'Colombo District — 48-hour advisory',
         transform=ax0.transAxes, fontsize=11, color='white', ha='center', alpha=0.9)
ax0.axis('off')

# Expected loss bars
ax1 = fig.add_subplot(gs[0, 2])
ax1.set_facecolor('white')
el = list(result['expected_losses'].values())
bars = ax1.barh([WARN_SHORT[i+1] for i in range(4)], el,
                color=[COLORS[i+1][0] for i in range(4)], alpha=0.85)
bars[w-1].set_edgecolor('black'); bars[w-1].set_linewidth(2.5)
ax1.set_xlabel('Expected loss')
ax1.set_title('Expected loss\nby warning level', fontsize=10, fontweight='bold')
ax1.invert_yaxis()

# Forecast probabilities
ax2 = fig.add_subplot(gs[1, 0])
ax2.set_facecolor('white')
pvals = list(result['p_by_state'].values())
ax2.bar(['Normal','Caution','Extreme','Danger'], pvals,
        color=[COLORS[i+1][0] for i in range(4)], alpha=0.85)
ax2.set_ylabel('Probability')
ax2.set_title('Calibrated probabilities', fontsize=10, fontweight='bold')
ax2.set_ylim(0, 1)
ax2.tick_params(axis='x', labelsize=9)

# Sector actions
ax3 = fig.add_subplot(gs[1, 1:])
ax3.axis('off')
ax3.set_facecolor('white')
ax3.set_title('Sector actions', fontsize=10, fontweight='bold', loc='left', pad=6)
sector_colors_brief = {'Health':'#C62828','Agri':'#2E7D32','Labour':'#E65100','Education':'#1565C0','DMC':'#6A1B9A'}
y = 0.96
for sector, actions in SECTOR_ACTIONS[w].items():
    color = sector_colors_brief.get(sector, '#333')
    ax3.text(0.0, y, f'[{sector}]', transform=ax3.transAxes,
             fontsize=9, fontweight='bold', color=color, va='top')
    y -= 0.09
    for action in actions[:2]:
        ax3.text(0.03, y, '• ' + action[:75] + ('…' if len(action)>75 else ''),
                 transform=ax3.transAxes, fontsize=8, color='#333', va='top')
        y -= 0.08
    if y < 0.05: break

plt.tight_layout()
plt.show()

---
## 9 · Threshold Refinement

The Clinic 5 plan notes that **further support is needed for threshold refinement**. This cell shows how the Bayesian system can help identify where the Watch/Alert/Warning switching points should be, based on observed data.

In [ ]:
sys_cal2 = HeatWarningSystem(sector='balanced', model_type='calibrated')
sys_cal2.fit(train_data)

p3_grid = np.linspace(0, 0.55, 30)
p4_grid = np.linspace(0, 0.55, 30)
warn_grid = np.zeros((len(p4_grid), len(p3_grid)), dtype=int)

for i, p4 in enumerate(p4_grid):
    for j, p3 in enumerate(p3_grid):
        p1 = max(0, (1-p3-p4)*0.6); p2 = max(0, 1-p1-p3-p4)
        fc = np.array([p1,p2,p3,p4]); fc /= fc.sum()
        warn_grid[i,j] = sys_cal2.issue_warning(fc)['warning']

cmap = matplotlib.colors.ListedColormap([COLORS[w][0] for w in [1,2,3,4]])
norm = matplotlib.colors.BoundaryNorm([0.5,1.5,2.5,3.5,4.5], cmap.N)

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.pcolormesh(p3_grid, p4_grid, warn_grid, cmap=cmap, norm=norm)
ax.contour(p3_grid, p4_grid, warn_grid, levels=[1.5,2.5,3.5],
           colors='white', linewidths=2, linestyles='--')
ax.set_xlabel('P(Cat 3 — Extreme Caution) in ensemble forecast', fontsize=12)
ax.set_ylabel('P(Cat 4 — Danger) in ensemble forecast', fontsize=12)
ax.set_title('Optimal Warning Decision Space\n'
             'Dashed lines = switching boundaries (refine thresholds here)',
             fontsize=13, fontweight='bold')

patches = [mpatches.Patch(color=COLORS[w][0], label=f'Level {w}: {WARN_SHORT[w]}') for w in [1,2,3,4]]
ax.legend(handles=patches, loc='upper left', fontsize=10, framealpha=0.9)
ax.text(0.52, 0.05, '← Refine these boundaries\nusing real Colombo data',
        transform=ax.transAxes, fontsize=10, color='white', style='italic',
        va='bottom', ha='left')

plt.tight_layout()
plt.show()

print('\nNext step: replace generate_demo_data() with real DoM Heat Index records')
print('and re-run this notebook. The boundary lines will shift to reflect')
print('Colombo climatology and your actual forecast ensemble.')

---
## 10 · Connecting to Real Data

Once the DoM has historical observations, the transition is:

```python
# 1. Create your CSV (use the template from heat_ews_demo.py --seasonal)
# 2. Load it here
import pandas as pd
df = pd.read_csv('colombo_2020_2025.csv')
obs_cats = np.array([categorise_hi(float(h)) for h in df['heat_index']])
# ... (probs as shown in Section 1 above)

# 3. Fit and evaluate
train = HeatData(obs_cats[:1500], probs[:1500])   # ~4 years
test  = HeatData(obs_cats[1500:], probs[1500:])   # ~1 year
sys = HeatWarningSystem(sector='health', model_type='calibrated')
sys.fit(train)
results = sys.evaluate(test)
print(f'POD: {results["POD"]:.2f}  FAR: {results["FAR"]:.2f}')

# 4. Feed results back into the HTML dashboard and SOP template
```

**Key adjustments to make with real data:**
- Update `HI_THRESHOLDS` in `heat_ews_srilanka.py` if the DoM uses different breakpoints
- Re-run this notebook for each sector (`health`, `agri`, `labour`) separately
- Use the FORIN review form (Tab 3 of the HTML tool / Section 5 of the SOP) after each heat season to update the contingency table

---
*Sri Lanka Heat IBF Demo Notebook · CREWS SA · March 2026*